# 📊 Análisis Exploratorio - Sistema ML Optimización de Químicos

Este notebook contiene análisis exploratorio de datos y visualizaciones para el sistema ML.

**Autor**: Sistema ML SICOP  
**Fecha**: 2024  
**Propósito**: Análisis de datos históricos y evaluación de modelos predictivos

## 1. Configuración Inicial

In [ ]:
# Imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Agregar directorio raíz al path
root_dir = Path.cwd().parent.parent
sys.path.insert(0, str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta

# Librerías del sistema ML
from core.database import SessionLocal
from ml.data.repository import PlantDataRepository
from ml.data.preprocessor import DataPreprocessor
from ml.features.feature_engineer import FeatureEngineer
from ml.models.model_manager import ModelManager
from ml.utils.config_manager import get_config

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Librerías importadas correctamente")

## 2. Carga de Datos

In [ ]:
# Conectar a base de datos
db = SessionLocal()
repository = PlantDataRepository(db)
config = get_config()

# Obtener estadísticas
stats = repository.get_data_statistics()

print("📊 Estadísticas de la Base de Datos")
print("=" * 50)
print(f"Registros operativos: {stats['total_operational_records']}")
print(f"Registros de consumo: {stats['total_consumption_records']}")
print(f"Rango de fechas: {stats['date_range']['start']} a {stats['date_range']['end']}")
print(f"Total operaciones fisicoquímicas: {stats['total_physicochemical_records']}")
print(f"Total producción filtros: {stats['total_production_records']}")

In [ ]:
# Cargar dataset completo
end_date = date.today()
start_date = end_date - timedelta(days=180)

df = repository.get_combined_dataset(start_date=start_date, end_date=end_date)

print(f"\n✓ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nColumnas disponibles:\n{list(df.columns)}")

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Información general
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Valores faltantes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores Faltantes': missing.values,
    'Porcentaje': missing_pct.values
})

missing_df = missing_df[missing_df['Valores Faltantes'] > 0].sort_values('Valores Faltantes', ascending=False)

if len(missing_df) > 0:
    print("⚠ Valores Faltantes:")
    print(missing_df.to_string(index=False))
else:
    print("✓ No hay valores faltantes")

### 3.1 Distribución de Variables

In [ ]:
# Variables de entrada principales
input_vars = ['turbedad_at', 'turbedad_ac', 'ph_at', 'ph_ac', 'temperatura', 
              'cloro_residual', 'conductividad_at', 'conductividad_ac']

fig, axes = plt.subplots(4, 2, figsize=(15, 12))
axes = axes.ravel()

for idx, var in enumerate(input_vars):
    if var in df.columns:
        axes[idx].hist(df[var].dropna(), bins=30, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribución: {var}', fontsize=10, fontweight='bold')
        axes[idx].set_xlabel(var)
        axes[idx].set_ylabel('Frecuencia')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Distribuciones visualizadas")

### 3.2 Variables Objetivo (Consumo de Químicos)

In [ ]:
# Variables objetivo
target_vars = ['sulfato_aluminio', 'cal', 'hipoclorito_calcio', 'cloro_gas']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, var in enumerate(target_vars):
    if var in df.columns:
        data = df[var].dropna()
        axes[idx].hist(data, bins=30, edgecolor='black', alpha=0.7, color='coral')
        axes[idx].set_title(f'Consumo: {var}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('kg/día')
        axes[idx].set_ylabel('Frecuencia')
        axes[idx].grid(True, alpha=0.3)
        
        # Estadísticas
        mean_val = data.mean()
        median_val = data.median()
        axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.2f}')
        axes[idx].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_val:.2f}')
        axes[idx].legend()

plt.tight_layout()
plt.show()

print("✓ Distribuciones de consumo visualizadas")

### 3.3 Correlaciones

In [ ]:
# Matriz de correlación
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación - Todas las Variables', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Matriz de correlación completa generada")

In [ ]:
# Correlación con variables objetivo
if all(var in df.columns for var in target_vars):
    target_corr = df[numeric_cols].corr()[target_vars].sort_values(by=target_vars[0], ascending=False)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(target_corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, 
                linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlación con Variables Objetivo (Consumo de Químicos)', 
              fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Top correlaciones con Sulfato de Aluminio:")
    print(target_corr['sulfato_aluminio'].head(10))

### 3.4 Series Temporales

In [ ]:
# Preparar datos temporales
if 'fecha' in df.columns:
    df_temporal = df.copy()
    df_temporal['fecha'] = pd.to_datetime(df_temporal['fecha'])
    df_temporal = df_temporal.sort_values('fecha')
    df_temporal.set_index('fecha', inplace=True)
    
    # Gráfico de turbidez
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))
    
    if 'turbedad_at' in df_temporal.columns and 'turbedad_ac' in df_temporal.columns:
        axes[0].plot(df_temporal.index, df_temporal['turbedad_at'], label='Agua Tratada (entrada)', alpha=0.7)
        axes[0].plot(df_temporal.index, df_temporal['turbedad_ac'], label='Agua Clarificada (salida)', alpha=0.7)
        axes[0].set_title('Evolución de Turbidez', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('NTU')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
    
    # Gráfico de pH
    if 'ph_at' in df_temporal.columns and 'ph_ac' in df_temporal.columns:
        axes[1].plot(df_temporal.index, df_temporal['ph_at'], label='pH Agua Tratada', alpha=0.7, color='blue')
        axes[1].plot(df_temporal.index, df_temporal['ph_ac'], label='pH Agua Clarificada', alpha=0.7, color='green')
        axes[1].axhline(7.0, color='red', linestyle='--', linewidth=1, label='pH neutro')
        axes[1].set_title('Evolución de pH', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('pH')
        axes[1].set_xlabel('Fecha')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Series temporales visualizadas")

In [ ]:
# Consumo de químicos en el tiempo
if all(var in df_temporal.columns for var in target_vars):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    
    for idx, (var, color) in enumerate(zip(target_vars, colors)):
        axes[idx].plot(df_temporal.index, df_temporal[var], color=color, alpha=0.7, linewidth=2)
        axes[idx].set_title(f'Consumo de {var}', fontsize=11, fontweight='bold')
        axes[idx].set_ylabel('kg/día')
        axes[idx].set_xlabel('Fecha')
        axes[idx].grid(True, alpha=0.3)
        
        # Media móvil
        rolling_mean = df_temporal[var].rolling(window=7).mean()
        axes[idx].plot(df_temporal.index, rolling_mean, color='red', linestyle='--', 
                      linewidth=2, label='Media móvil (7 días)', alpha=0.8)
        axes[idx].legend()
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Evolución temporal de consumos visualizada")

## 4. Feature Engineering

In [ ]:
# Aplicar feature engineering
print("🔧 Aplicando Feature Engineering...")
print(f"Columnas antes: {df.shape[1]}")

df_engineered = FeatureEngineer.engineer_features(
    df,
    create_ratios=True,
    create_deltas=True,
    create_interactions=True,
    create_temporal=True
)

print(f"Columnas después: {df_engineered.shape[1]}")
print(f"Features creadas: {df_engineered.shape[1] - df.shape[1]}")

# Mostrar nuevas features
new_features = [col for col in df_engineered.columns if col not in df.columns]
print(f"\nNuevas features: {new_features}")

## 5. Evaluación de Modelo

In [ ]:
# Cargar modelo entrenado
model_manager = ModelManager()
available_models = model_manager.list_available_models()

if len(available_models) > 0:
    print("📦 Modelos disponibles:")
    for model_info in available_models:
        print(f"  - {model_info['filename']} ({model_info['date']})")
    
    # Cargar el más reciente
    latest_path = model_manager.get_latest_model_path()
    model_data = model_manager.load_model(latest_path)
    
    print(f"\n✓ Modelo cargado: {latest_path.name}")
    print(f"Tipo: {model_data['metadata']['model_type']}")
    print(f"Entrenado: {model_data['metadata']['trained_at']}")
    
    # Métricas
    if 'metrics' in model_data['metadata']:
        metrics = model_data['metadata']['metrics']
        print(f"\n📊 Métricas del modelo:")
        print(f"  R² Score: {metrics.get('r2', 'N/A')}")
        print(f"  RMSE: {metrics.get('rmse', 'N/A')}")
        print(f"  MAE: {metrics.get('mae', 'N/A')}")
        print(f"  MAPE: {metrics.get('mape', 'N/A')}%")
        
else:
    print("⚠ No hay modelos entrenados disponibles.")
    print("Ejecute: python train_ml_model.py")

In [ ]:
# Feature Importance
if len(available_models) > 0 and 'feature_importance' in model_data['metadata']:
    importance_df = pd.DataFrame(model_data['metadata']['feature_importance'])
    importance_df = importance_df.sort_values('importance', ascending=False).head(20)
    
    plt.figure(figsize=(12, 8))
    plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue', alpha=0.8)
    plt.xlabel('Importancia', fontsize=12)
    plt.title('Top 20 Features Más Importantes', fontsize=14, fontweight='bold', pad=15)
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print("✓ Importancia de features visualizada")
    print(f"\nTop 10 features:")
    print(importance_df.head(10).to_string(index=False))

## 6. Predicciones de Ejemplo

In [ ]:
# Hacer predicción de ejemplo
from ml.inference.predictor_service import ChemicalConsumptionPredictor

predictor = ChemicalConsumptionPredictor()

# Datos de ejemplo
example_data = {
    'turbedad_at': 50.0,
    'turbedad_ac': 5.0,
    'ph_at': 7.2,
    'ph_ac': 7.5,
    'temperatura': 22.0,
    'cloro_residual': 0.8,
    'conductividad_at': 400.0,
    'conductividad_ac': 380.0,
    'caudal': 250.0,
    'presion_entrada': 3.0,
    'presion_salida': 2.5,
    'solidos_totales': 200.0
}

try:
    result = predictor.predict(example_data)
    
    print("🎯 Predicción de Consumo de Químicos")
    print("=" * 50)
    print(f"Sulfato de Aluminio: {result.sulfato_aluminio:.2f} kg/día")
    print(f"Cal: {result.cal:.2f} kg/día")
    print(f"Hipoclorito de Calcio: {result.hipoclorito_calcio:.2f} kg/día")
    print(f"Cloro Gas: {result.cloro_gas:.2f} kg/día")
    print(f"\nConfianza: {result.confidence:.2%}")
    print(f"Costo Estimado: ${result.estimated_cost:.2f}/día")
    print(f"Modelo: {result.model_version}")
    
    # Visualizar predicción
    chemicals = ['Sulfato\nAluminio', 'Cal', 'Hipoclorito\nCalcio', 'Cloro\nGas']
    values = [result.sulfato_aluminio, result.cal, result.hipoclorito_calcio, result.cloro_gas]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(chemicals, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    plt.ylabel('Consumo (kg/día)', fontsize=12)
    plt.title('Predicción de Consumo Diario de Químicos', fontsize=14, fontweight='bold', pad=15)
    plt.grid(True, alpha=0.3, axis='y')
    
    # Agregar valores encima de las barras
    for bar, value in zip(bars, values):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.1f} kg',
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"❌ Error en predicción: {e}")
    print("Asegúrese de haber entrenado un modelo primero: python train_ml_model.py")

## 7. Detección de Anomalías

In [ ]:
# Detectar anomalías en datos recientes
from ml.inference.anomaly_service import AnomalyDetectorService

anomaly_detector = AnomalyDetectorService()

try:
    # Entrenar detector si no existe
    operational_data = repository.get_operational_data(
        start_date=start_date,
        end_date=end_date
    )
    
    if len(operational_data) > 30:
        anomaly_detector.train_detector(operational_data)
        print("✓ Detector de anomalías entrenado")
        
        # Detectar anomalías
        anomalies = anomaly_detector.detect_anomalies(operational_data[-30:])
        
        anomaly_count = sum(1 for a in anomalies if a.is_anomaly)
        print(f"\n🔍 Anomalías detectadas: {anomaly_count} de {len(anomalies)} registros")
        
        # Mostrar anomalías críticas
        critical = [a for a in anomalies if a.severity == 'critico']
        if critical:
            print(f"\n⚠ Anomalías CRÍTICAS: {len(critical)}")
            for anomaly in critical[:5]:
                print(f"  - Fecha: {anomaly.timestamp}")
                print(f"    Score: {anomaly.anomaly_score:.3f}")
                print(f"    Razones: {', '.join(anomaly.reasons)}")
                print()
    else:
        print("⚠ Datos insuficientes para detección de anomalías (mínimo 30 registros)")
        
except Exception as e:
    print(f"❌ Error en detección de anomalías: {e}")

## 8. Resumen y Conclusiones

In [ ]:
print("📋 RESUMEN DEL ANÁLISIS")
print("=" * 70)
print(f"\n1. DATOS")
print(f"   - Total registros: {len(df)}")
print(f"   - Rango temporal: {stats['date_range']['start']} a {stats['date_range']['end']}")
print(f"   - Variables: {df.shape[1]} originales, {df_engineered.shape[1]} con features")

if len(available_models) > 0:
    print(f"\n2. MODELO")
    print(f"   - Tipo: {model_data['metadata']['model_type']}")
    metrics = model_data['metadata'].get('metrics', {})
    print(f"   - R² Score: {metrics.get('r2', 'N/A')}")
    print(f"   - RMSE: {metrics.get('rmse', 'N/A')}")
    print(f"   - MAE: {metrics.get('mae', 'N/A')}")

print(f"\n3. CALIDAD DE DATOS")
print(f"   - Completitud: {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.1f}%")
print(f"   - Variables numéricas: {len(numeric_cols)}")

print(f"\n4. RECOMENDACIONES")
if stats['total_operational_records'] < 180:
    print("   ⚠ Recolectar más datos históricos para mejorar precisión")
if len(missing_df) > 0:
    print("   ⚠ Completar valores faltantes en variables críticas")
if len(available_models) > 0 and metrics.get('r2', 0) < 0.80:
    print("   ⚠ Considerar más feature engineering o datos adicionales")
print("   ✓ Monitorear anomalías regularmente")
print("   ✓ Re-entrenar modelo mensualmente con nuevos datos")

print("\n" + "=" * 70)
print("✅ Análisis completado")

In [ ]:
# Cerrar conexión
db.close()
print("\n✓ Conexión a base de datos cerrada")